<a href="https://colab.research.google.com/github/Tamaki-Baba/data-Econometrics/blob/main/%E3%83%86%E3%82%99%E3%83%BC%E3%82%BF%E3%81%A8%E8%A8%88%E9%87%8F%E7%B5%8C%E6%B8%88%E5%AD%A6_1204.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### ナイーブベイズを用いたスパムメール判定をするプログラム

In [ ]:
import math
import numpy as np

try:
    import MeCab
    MECAB_AVAILABLE = True
except:
    MECAB_AVAILABLE = False

def tokenize(text):
    """MeCabが使えない場合は簡易トークナイズ"""
    if MECAB_AVAILABLE:
        tagger = MeCab.Tagger("-Owakati")
        parsed = tagger.parse(text)
        if parsed is None:
            return []
        return parsed.strip().split()
    else:
        import re
        tokens = re.findall(r"[A-Za-z0-9]+", text.lower())
        return tokens


train_texts = [
    "Win a free lottery now claim your prize",
    "Limited time offer click here to buy",
    "Meeting tomorrow please prepare document",
    "Lunch at 12 meet in the lobby",
    "Cheap meds available no prescription needed",
    "Important update your account information",
    "Project final report attached",
    "Re schedule and tasks"
]

train_labels = [1,1,0,0,1,1,0,0]  # 1=spam, 0=ham

from collections import Counter

def build_counts(texts, labels):
    vocab = set()
    word_counts = {0: Counter(), 1: Counter()}
    doc_count = {0: 0, 1: 0}

    for text, lab in zip(texts, labels):
        doc_count[lab] += 1
        tokens = tokenize(text)
        vocab.update(tokens)
        word_counts[lab].update(tokens)

    return vocab, word_counts, doc_count


vocab, word_counts, doc_count = build_counts(train_texts, train_labels)


def wordProb(word, cls, word_counts, vocab, alpha=1.0):
    """
    P(word | cls) をラプラス平滑で計算
    """
    total_words = sum(word_counts[cls].values())
    count_w = word_counts[cls].get(word, 0)
    V = len(vocab)

    prob = (count_w + alpha) / (total_words + alpha * V)
    return prob

def score(text, priors, word_counts, vocab, alpha=1.0):
    tokens = tokenize(text)
    scores = {}

    for cls in [0, 1]:
        # P(class) の log
        log_prob = math.log(priors[cls])

        # Σ log(P(word|class))
        for t in tokens:
            p = wordProb(t, cls, word_counts, vocab, alpha)
            log_prob += math.log(p)

        scores[cls] = log_prob

    # 最大スコアのクラス
    pred = max(scores, key=scores.get)
    return scores, pred


def train_naive_bayes(texts, labels, alpha=1.0):
    vocab, word_counts, doc_count = build_counts(texts, labels)

    total_docs = sum(doc_count.values())
    priors = {
        0: doc_count[0] / total_docs,
        1: doc_count[1] / total_docs
    }

    return {
        "vocab": vocab,
        "word_counts": word_counts,
        "priors": priors,
        "alpha": alpha
    }


model = train_naive_bayes(train_texts, train_labels, alpha=1.0)

correct = 0

for text, lab in zip(train_texts, train_labels):
    scores, pred = score(
        text,
        model["priors"],
        model["word_counts"],
        model["vocab"],
        model["alpha"]
    )
    if pred == lab:
        correct += 1
    print(f"{text}\n  True={lab}, Pred={pred}, Score={scores}\n")

print("Accuracy =", correct / len(train_texts))

test_mail = "Congratulations! You won a prize click to claim"
scores, pred = score(
    test_mail,
    model["priors"],
    model["word_counts"],
    model["vocab"],
    model["alpha"]
)
print("Test:", test_mail)
print("Predicted:", pred)
print("Scores:", scores)


Win a free lottery now claim your prize
  True=1, Pred=1, Score={0: -34.088245339725034, 1: -28.843943644302747}

Limited time offer click here to buy
  True=1, Pred=1, Score={0: -29.9138580698294, 1: -25.67987605592954}

Meeting tomorrow please prepare document
  True=0, Pred=0, Score={0: -18.099347627238405, 1: -22.00654656576652}

Lunch at 12 meet in the lobby
  True=0, Pred=0, Score={0: -25.06182780590979, 1: -30.53190631984915}

Cheap meds available no prescription needed
  True=1, Pred=1, Score={0: -25.739470799933763, 1: -22.11034335944817}

Important update your account information
  True=1, Pred=1, Score={0: -21.565083530038127, 1: -18.13534555485863}

Project final report attached
  True=0, Pred=0, Score={0: -14.618107537902713, 1: -17.743866688725205}

Re schedule and tasks
  True=0, Pred=0, Score={0: -14.618107537902713, 1: -17.743866688725205}

Accuracy = 1.0
Test: Congratulations! You won a prize click to claim
Predicted: 1
Scores: {0: -34.088245339725034, 1: -31.32885029